# INV-M365-H Config Authority

Purpose: prove the precedence model required for `B1` runtime config authority remediation.

Lemma mapping: `L6`.
Invariant support: tenant-selected production authority must resolve from tenant config plus injected secret fallback, while bootstrap dotenv inputs remain non-authoritative.
Expected result: the precedence and bootstrap-filter logic is deterministic and replay-stable.


In [ ]:
AUTHORITY_KEYS = {
    'AZURE_TENANT_ID', 'GRAPH_TENANT_ID', 'MICROSOFT_TENANT_ID',
    'AZURE_CLIENT_ID', 'AZURE_APP_CLIENT_ID_TAI', 'GRAPH_CLIENT_ID', 'MICROSOFT_CLIENT_ID',
    'AZURE_CLIENT_SECRET', 'AZURE_APP_CLIENT_SECRET_TAI', 'GRAPH_CLIENT_SECRET', 'MICROSOFT_CLIENT_SECRET',
    'AZURE_CLIENT_CERTIFICATE_PATH', 'SHAREPOINT_HOSTNAME', 'SP_HOSTNAME'
}

def bootstrap_values(dotenv_values, existing_env):
    tenant_selected = bool((existing_env.get('UCP_TENANT') or dotenv_values.get('UCP_TENANT') or '').strip())
    loaded = {}
    for key, value in dotenv_values.items():
        if value in (None, ''):
            continue
        if key in existing_env:
            continue
        if tenant_selected and key in AUTHORITY_KEYS:
            continue
        loaded[key] = value
    return loaded

def resolve_graph_auth(tenant_cfg, env_cfg):
    return {
        'tenant_id': tenant_cfg.get('tenant_id') or env_cfg.get('tenant_id') or '',
        'client_id': tenant_cfg.get('client_id') or env_cfg.get('client_id') or '',
        'client_secret': tenant_cfg.get('client_secret') or env_cfg.get('client_secret') or '',
        'client_certificate_path': tenant_cfg.get('client_certificate_path') or env_cfg.get('client_certificate_path') or '',
    }


## Deterministic Assertions

The assertions below capture the required production-vs-bootstrap boundary and the tenant-config precedence rule.


In [ ]:
tenant_selected_dotenv = {
    'UCP_TENANT': 'pilot',
    'AZURE_TENANT_ID': 'dotenv-tenant',
    'AZURE_CLIENT_ID': 'dotenv-client',
    'AZURE_CLIENT_SECRET': 'dotenv-secret',
    'M365_SERVER_PORT': '9000'
}
loaded = bootstrap_values(tenant_selected_dotenv, {})
assert loaded['UCP_TENANT'] == 'pilot'
assert loaded['M365_SERVER_PORT'] == '9000'
assert 'AZURE_TENANT_ID' not in loaded
assert 'AZURE_CLIENT_ID' not in loaded
assert 'AZURE_CLIENT_SECRET' not in loaded

local_dotenv = {
    'AZURE_TENANT_ID': 'local-tenant',
    'AZURE_CLIENT_ID': 'local-client',
    'AZURE_CLIENT_SECRET': 'local-secret'
}
loaded_local = bootstrap_values(local_dotenv, {})
assert loaded_local['AZURE_TENANT_ID'] == 'local-tenant'
assert loaded_local['AZURE_CLIENT_ID'] == 'local-client'
assert loaded_local['AZURE_CLIENT_SECRET'] == 'local-secret'

resolved = resolve_graph_auth(
    {
        'tenant_id': 'tenant-yaml-tenant',
        'client_id': 'tenant-yaml-client',
        'client_secret': '',
        'client_certificate_path': '/secure/cert.pem'
    },
    {
        'tenant_id': 'env-tenant',
        'client_id': 'env-client',
        'client_secret': 'env-secret',
        'client_certificate_path': ''
    }
)
assert resolved['tenant_id'] == 'tenant-yaml-tenant'
assert resolved['client_id'] == 'tenant-yaml-client'
assert resolved['client_secret'] == 'env-secret'
assert resolved['client_certificate_path'] == '/secure/cert.pem'

replayed = resolve_graph_auth(
    {
        'tenant_id': 'tenant-yaml-tenant',
        'client_id': 'tenant-yaml-client',
        'client_secret': '',
        'client_certificate_path': '/secure/cert.pem'
    },
    {
        'tenant_id': 'env-tenant',
        'client_id': 'env-client',
        'client_secret': 'env-secret',
        'client_certificate_path': ''
    }
)
assert resolved == replayed
print('INV-M365-H assertions passed')
